# Course–Job Embedding Runner

**What this does:** Clones the repo and runs the embedding pipeline on Colab's GPU.

**Setup:** Go to Runtime > Change runtime type > T4 GPU

**All logic lives in `src/embedding/embed.py`** — this notebook is just a launcher.

In [1]:
# 1. Clone the repo
REPO_URL = "https://github.com/DanDmc/DSA4264_Project.git"  # <-- CHANGE THIS IF NEEDED
REPO_NAME = REPO_URL.split("/")[-1].replace(".git", "")

!git clone {REPO_URL}
%cd {REPO_NAME}

Cloning into 'DSA4264_Project'...
remote: Enumerating objects: 24344, done.
remote: Counting objects: 100% (14/14), done.
remote: Compressing objects: 100% (14/14), done.
remote: Total 24344 (delta 0), reused 10 (delta 0), pack-reused 24330 (from 2)
Receiving objects: 100% (24344/24344), 117.16 MiB | 16.91 MiB/s, done.
Resolving deltas: 100% (10695/10695), done.
Updating files: 100% (22762/22762), done.
/content/DSA4264_Project


In [2]:
# 2. Install dependencies
!pip install -q sentence-transformers pandas numpy torch

In [3]:
# 3. Run the embedding pipeline
!python src/embedding/embed.py

COURSE–JOB EMBEDDING PIPELINE

Loading modules from: /content/DSA4264_Project/data/processed/cleaned_modules.csv
  → 14,194 modules
Loading jobs from: /content/DSA4264_Project/data/processed/final_jobs_processed_filtered.csv
  → 13,663 jobs

Sample module text (truncated):
  Represent this university course for matching to relevant job positions: Biomedical Innovation & Enterprise. furnish students with a thorough understanding of a bio-venture from research and developme...

Sample job text (truncated):
  Represent this job posting for matching to relevant university courses: Sales Administrator. - Serve as administrator / key operator for all sales related systems i.e.
Amadeus Advance, Opera PMS, Lany...
Loading model: BAAI/bge-large-en-v1.5
Device: cuda (Tesla T4)
modules.json: 100% 349/349 [00:00<00:00, 501kB/s]
config_sentence_transformers.json: 100% 124/124 [00:00<00:00, 787kB/s]
README.md: 94.6kB [00:00, 44.5MB/s]
sentence_bert_config.json: 100% 52.0/52.0 [00:00<00:00, 363kB/s]


In [4]:
# 4. (Optional) Copy outputs to Google Drive so they persist after Colab disconnects
# Run the lines below if you want to save to Drive before committing to git.

from google.colab import drive
drive.mount('/content/drive')
!cp -r data/embeddings/ /content/drive/MyDrive/embeddings_backup/

Mounted at /content/drive


In [7]:
# 5. (Optional) Cast embeddings to smaller storage size and save locally to manually push to git
#cast emebeddings to float16 instead of float32 to reduce size and get under github's 50mb limit
import numpy as np

# Load, cast to float16, save back
for name in ["module_embeddings_bge-large-en-v1.5.npy", "job_embeddings_bge-large-en-v1.5.npy"]:
    path = f"data/embeddings/{name}"
    arr = np.load(path)
    np.save(path, arr.astype(np.float16))
    size_mb = arr.nbytes / (1024**2)
    new_size_mb = arr.astype(np.float16).nbytes / (1024**2)
    print(f"{name}: {size_mb:.1f}MB → {new_size_mb:.1f}MB")

#manually download data/embeddings folder to push to git manually
!zip -r /content/embeddings.zip /content/DSA4264_Project/data/embeddings/
from google.colab import files
files.download('/content/embeddings.zip')

module_embeddings_bge-large-en-v1.5.npy: 55.4MB → 27.7MB
job_embeddings_bge-large-en-v1.5.npy: 53.4MB → 26.7MB
updating: content/DSA4264_Project/data/embeddings/ (stored 0%)
updating: content/DSA4264_Project/data/embeddings/module_index.csv (deflated 85%)
updating: content/DSA4264_Project/data/embeddings/embedding_config.json (deflated 50%)
updating: content/DSA4264_Project/data/embeddings/job_index.csv (deflated 79%)
updating: content/DSA4264_Project/data/embeddings/module_embeddings_bge-large-en-v1.5.npy (deflated 22%)
updating: content/DSA4264_Project/data/embeddings/job_embeddings_bge-large-en-v1.5.npy (deflated 11%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# 6. (Optional) Commit embeddings back to the repo from Colab
# Only works if you've set up git credentials in Colab.
# easier to manually download data/embeddings folder and push to git

# !git lfs install
# !git lfs track "*.npy"
# !git add data/embeddings/
# !git commit -m "Add precomputed embeddings (bge-large-en-v1.5)"
# !git push